# Scraping de todos los equipos — Liga Profesional Argentina

Baja los partidos de **los 30 equipos de la Liga Profesional 2026** (torneo aparte de la
Primera Nacional donde juega Chaco For Ever — no comparte `tournament_id` ni equipos),
reutilizando la misma infraestructura de scraping que Primera Nacional pero apuntando a su
propio árbol de datos y salidas: `src/scraping/lpf_scraper.py`.

Para Primera Nacional (Zona A + Zona B, los equipos de Chaco) usar el notebook gemelo:
`notebooks/primera_nacional/scraping_todos_los_equipos.ipynb`.

**Cuándo correrlo:** una vez terminada cada fecha. Como el scraper es incremental (los partidos
que ya están completos se saltean sin llamadas de red), cada corrida solo baja lo nuevo.

**Para qué:** tener los datos de los 30 equipos de LPF ya en disco para armar estadísticas
comparativas de la liga y, más adelante, el mismo tipo de informe estándar por equipo que ya
existe para Primera Nacional (`informe_equipo_liga.ipynb`).

**Qué baja de cada partido:** lo mismo que un partido de For Ever — los 9 endpoints crudos
(`event`, `statistics`, `lineups`, `incidents`, `shotmap`, `average_positions`, `graph`,
`best_players`, `heatmaps`) más el DT (`managers`), y los 9 CSV limpios.

Guarda en `data/liga_profesional/{season}/equipos/{slug}/` (en vez de
`data/temporadas/{season}/rivals/{slug}/`, que es de Primera Nacional) y los gráficos en
`outputs/liga_profesional/{season}/equipos/{slug}/graficos/`.

Las pausas entre requests son las de siempre: `config.MIN_DELAY`–`config.MAX_DELAY` segundos.

> **Ojo con la primera corrida.** Son 30 equipos x ~20-30 partidos cada uno. Es una corrida
> grande (varias horas), pensada para dejar en background — ver `scripts/scrape_overnight.py`,
> que ya hace exactamente esto (`scrape_team_lpf` sobre los 30 equipos) como parte de la corrida
> nocturna. Este notebook sirve para correrlo a mano, en un subconjunto, o para el análisis de
> temporada de la sección 4 en adelante (que sí conviene correr acá).

In [ ]:
import os, sys, time, random

%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) in ('primera_nacional', 'liga_profesional'):
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import config
from src.scraping.lpf_scraper import scrape_team_lpf, scrape_managers_lpf

# ============================================================
# CONFIGURAR ACÁ
SEASON   = '2026'
EQUIPOS  = list(config.LPF_TEAMS_2026)   # los 30 equipos de la Liga Profesional
CON_DT   = True                          # bajar tambien el DT de cada partido (/managers)
HEATMAPS = True                          # mapas de calor por jugador (~22 requests extra por partido)
HEADLESS = False                         # False = con ventana, igual que los notebooks de Primera Nacional
QUICK    = True                          # corrida post-fecha: solo las últimas 2 páginas del feed
                                         # de cada equipo. False para la PRIMERA corrida / backfill.
RETRY_EMPTY = False                      # True para reintentar endpoints que SofaScore no había publicado
# ============================================================

print(f'Liga Profesional {SEASON} · {len(EQUIPOS)} equipos')
print(f'Modo: {"QUICK (post-fecha)" if QUICK else "COMPLETO (backfill)"}')
print(f'Pausa entre endpoints: {config.MIN_DELAY_EP}-{config.MAX_DELAY_EP} s · entre partidos: {config.MIN_DELAY}-{config.MAX_DELAY} s')
for s in EQUIPOS:
    print(f'  · {s:26s} {config.LPF_TEAMS_2026[s]["name"]}')


## 1. Qué falta bajar

Antes de disparar nada, mirá cuántos partidos hay ya en disco por equipo.

In [ ]:
from src.scraping.chaco_scraper import _match_done

def partidos_en_disco(slug, season=SEASON):
    """(completos, incompletos), igual criterio que Primera Nacional pero
    apuntando a la carpeta propia de LPF."""
    d = config.lpf_team_dir(slug, season)
    if not os.path.isdir(d):
        return 0, 0
    dirs = [os.path.join(d, m) for m in os.listdir(d) if os.path.isdir(os.path.join(d, m))]
    ok = sum(1 for m in dirs if _match_done(m))
    return ok, len(dirs) - ok

print(f'{"EQUIPO":30s} {"COMPLETOS":>9s} {"A REBAJAR":>9s}')
for slug in EQUIPOS:
    ok, falta = partidos_en_disco(slug)
    print(f'{config.LPF_TEAMS_2026[slug]["name"]:30s} {ok:>9d} {falta:>9d}')

## 2. Scraping de todos los equipos

Loop sobre los 30 equipos con `scrape_team_lpf` / `scrape_managers_lpf` (los wrappers de
`lpf_scraper.py` que reusan `scrape_rival_recent`/`scrape_rival_managers` apuntando al árbol de
datos de LPF). `n_matches=None` baja todos los partidos jugados de la temporada.

Es reanudable: si se corta a mitad, volvés a correr la celda y sigue donde estaba.

In [ ]:
resumen = {}
fallidos = {}
sep = '=' * 70

for i, slug in enumerate(EQUIPOS, 1):
    nombre = config.LPF_TEAMS_2026[slug]['name']
    print(f'\n{sep}\n[{i}/{len(EQUIPOS)}] {nombre}  ({slug})\n{sep}')
    try:
        mks = scrape_team_lpf(slug, season=SEASON, headless=HEADLESS,
                              include_heatmaps=HEATMAPS, n_matches=None,
                              quick=QUICK, retry_empty=RETRY_EMPTY)
        if CON_DT:
            scrape_managers_lpf(slug, season=SEASON, headless=HEADLESS)
        resumen[slug] = len(mks)
    except Exception as e:
        fallidos[slug] = repr(e)
        print(f'  ERROR en {nombre}: {e}')

    if i < len(EQUIPOS):
        time.sleep(random.uniform(config.MIN_DELAY, config.MAX_DELAY))

print(f'\n\nListo. {len(resumen)}/{len(EQUIPOS)} equipos ok.')
if fallidos:
    print('Fallidos (volvé a correr la celda: arranca donde quedó):')
    for slug, err in fallidos.items():
        print(f'  · {slug}: {err}')


## 3. Control final

Cuántos partidos quedó con cada equipo. Corriendo esto después de cada fecha, el número debería
subir de a uno por equipo.

In [ ]:
ARCHIVOS = ['event_clean.csv', 'statistics_clean.csv', 'lineups_clean.csv',
            'incidents_clean.csv', 'shotmap_clean.csv', 'average_positions_clean.csv',
            'momentum_clean.csv', 'best_players_clean.csv', 'player_heatmaps_clean.csv']

print(f'{"EQUIPO":30s} {"COMPLETOS":>9s} {"HEATMAP":>7s} {"DT":>4s}')
total = 0
for slug in EQUIPOS:
    d = config.lpf_team_dir(slug, SEASON)
    ok, _ = partidos_en_disco(slug)
    hm = dt = 0
    if os.path.isdir(d):
        for m in os.listdir(d):
            mp = os.path.join(d, m)
            hm += os.path.exists(os.path.join(mp, 'player_heatmaps_clean.csv'))
            dt += os.path.exists(os.path.join(mp, 'raw', 'managers.json'))
    total += ok
    print(f'{config.LPF_TEAMS_2026[slug]["name"]:30s} {ok:>9d} {hm:>7d} {dt:>4d}')
print(f'\nTotal de partidos completos: {total}')

faltantes = []
for slug in EQUIPOS:
    d = config.lpf_team_dir(slug, SEASON)
    if not os.path.isdir(d):
        continue
    for m in sorted(os.listdir(d)):
        mp = os.path.join(d, m)
        falta = [f for f in ARCHIVOS if not os.path.exists(os.path.join(mp, f))]
        if falta:
            faltantes.append((slug, m, falta))
if faltantes:
    print(f'\n{len(faltantes)} partidos con archivos faltantes:')
    for slug, m, falta in faltantes[:20]:
        print(f'  · {slug}/{m}: {", ".join(falta)}')
else:
    print('Todos los partidos tienen los 9 CSV.')

In [ ]:
# Partidos que no se pudieron scrapear del todo (solo LPF).
from src.scraping.audit import print_audit
_ = print_audit(SEASON, scope='lpf')


## 4. Análisis de temporada de cada equipo

Las funciones de `src/analysis/season.py` son genéricas: toman `match_keys` + `base_dir`, así que
corriendo lo mismo apuntando a `data/liga_profesional/{season}/equipos/{slug}/` sale el análisis
de temporada del equipo de LPF, igual que para un rival de Primera Nacional.

Los gráficos van a `outputs/liga_profesional/{season}/equipos/{slug}/graficos/`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
from IPython.display import Image, display

from src.analysis.season import (
    build_season_table, season_stats, season_player_stats,
    top_scorers, most_used_players,
    threat_by_situation, markov_game_state, goals_by_time_window,
    is_home_match,
)
from src.visualization.season import (
    save_fig, plot_season_points, plot_xg_season, plot_top_players,
    plot_shotmap_4grid, plot_recent_form, plot_home_away_panel,
    plot_discipline, plot_threat_by_situation, plot_markov_transitions,
    plot_goals_by_time_window,
)

RATING_MIN_MIN = 450   # minutos minimos para el top de rating

def show(path):
    if path and os.path.exists(path):
        display(Image(filename=path))

def match_keys_lpf(slug, season=SEASON):
    """Partidos completos del equipo de LPF, en orden cronologico."""
    d = config.lpf_team_dir(slug, season)
    if not os.path.isdir(d):
        return []
    return sorted(m for m in os.listdir(d) if _match_done(os.path.join(d, m)))

def analisis_temporada_equipo_lpf(slug, season=SEASON, graficos=True):
    """Analisis de temporada de un equipo de LPF, con las MISMAS funciones
    que el de For Ever / los rivales de Primera Nacional.

    Devuelve (stats, season_df, player_df). Graficos ->
    outputs/liga_profesional/{season}/equipos/{slug}/graficos/
    """
    nombre   = config.LPF_TEAMS_2026[slug]['name']
    base_dir = config.lpf_team_dir(slug, season)
    mks      = match_keys_lpf(slug, season)
    if not mks:
        print(f'{nombre}: sin partidos completos, se saltea.')
        return None, None, None

    season_df = build_season_table(mks, base_dir)
    stats     = season_stats(season_df)
    player_df = season_player_stats(mks, base_dir)

    if not graficos:
        return stats, season_df, player_df

    plots_dir = config.lpf_team_plots_dir(slug, season)
    os.makedirs(plots_dir, exist_ok=True)
    p = lambda f: os.path.join(plots_dir, f)

    save_fig(plot_season_points(season_df), p('puntos_acumulados.png'))

    fig = plot_xg_season(season_df)
    if fig:
        save_fig(fig, p('xg_temporada.png'))

    plot_recent_form(season_df, window=5, season=season, save_path=p('forma_reciente.png'))
    plot_home_away_panel(season_df, season=season, save_path=p('local_visitante.png'))

    if not player_df.empty:
        save_fig(plot_top_players(player_df, metric='minutos', top_n=15), p('top_minutos.png'))

        rating_df = player_df[player_df['minutos'] >= RATING_MIN_MIN].dropna(subset=['rating_promedio'])
        if not rating_df.empty:
            xmin = max(4.0, float(rating_df['rating_promedio'].min()) - 0.3)
            save_fig(plot_top_players(rating_df, metric='rating_promedio', top_n=15, xmin=xmin),
                     p('top_rating.png'))

        plot_discipline(player_df, season=season, save_path=p('disciplina.png'))

    shots = []
    for mk in mks:
        sp = os.path.join(base_dir, mk, 'shotmap_clean.csv')
        if os.path.exists(sp):
            df_s = pd.read_csv(sp)
            shots.append(df_s[df_s['isHome'] == is_home_match(mk)])
    if shots:
        df_shots = pd.concat(shots, ignore_index=True)
        save_fig(plot_shotmap_4grid(df_shots, team_label=nombre,
                                    title=f'Distribucion de Tiros - {nombre} {season}'),
                 p('shotmap_temporada.png'))

    thr = threat_by_situation(mks, base_dir)
    if thr is not None and not thr.empty:
        plot_threat_by_situation(thr, season=season, save_path=p('amenaza_por_situacion.png'))

    matrix_df, transitions_df = markov_game_state(mks, base_dir)
    if matrix_df is not None and not matrix_df.empty:
        plot_markov_transitions(matrix_df, transitions_df, season=season,
                                save_path=p('markov_estado_marcador.png'))

    win_df = goals_by_time_window(mks, base_dir)
    if win_df is not None and not win_df.empty:
        plot_goals_by_time_window(win_df, season=season, save_path=p('goles_por_ventana.png'))

    return stats, season_df, player_df

print('analisis_temporada_equipo_lpf() listo.')

In [ ]:
# Corre el analisis de temporada de los 30 equipos y arma la tabla comparativa
# "Anual" (todos contra todos, sin zonas — a diferencia de Primera Nacional,
# LPF sí tiene un unico ranking de los 30 que tiene sentido ordenar junto).
# Es solo lectura de disco + graficos: no toca la red, se puede repetir cuantas veces quieras.
temporadas = {}
filas = []

for slug in EQUIPOS:
    nombre = config.LPF_TEAMS_2026[slug]['name']
    stats, season_df, player_df = analisis_temporada_equipo_lpf(slug, SEASON)
    if not stats:
        continue
    temporadas[slug] = {'stats': stats, 'season_df': season_df, 'player_df': player_df}
    filas.append({
        'equipo': nombre,
        'PJ':  stats['partidos'],
        'V':   stats['victorias'],
        'E':   stats['empates'],
        'D':   stats['derrotas'],
        'Pts': stats['puntos'],
        'GF':  stats['goles_favor'],
        'GC':  stats['goles_contra'],
        'DG':  stats['diferencia_goles'],
        'xGF': stats['xg_favor'],
        'xGC': stats['xg_contra'],
        'local':      stats['rendimiento_local'],
        'visitante':  stats['rendimiento_visitante'],
    })
    print(f"OK {nombre:30s} {stats['partidos']:>2d} PJ | {stats['puntos']:>2d} pts | "
          f"{stats['goles_favor']}-{stats['goles_contra']}")

liga = pd.DataFrame(filas).sort_values(['Pts', 'DG', 'GF'], ascending=False).reset_index(drop=True)
liga.index += 1
print('\n=== La Liga Profesional segun nuestros propios datos ===')
print(liga.to_string())

liga_csv = os.path.join(config.lpf_outputs_dir(SEASON), 'liga_resumen.csv')
os.makedirs(os.path.dirname(liga_csv), exist_ok=True)
liga.to_csv(liga_csv, index=False, encoding='utf-8')
print(f'\nGuardado: {liga_csv}')

## 5. Un equipo en detalle

Poné el slug de un equipo y mirá su temporada acá mismo: tabla, goleadores, más utilizados y los
gráficos que se acaban de generar.

In [ ]:
EQUIPO_DETALLE = 'boca_juniors'    # slug del equipo a mirar

nombre = config.LPF_TEAMS_2026[EQUIPO_DETALLE]['name']
d = temporadas.get(EQUIPO_DETALLE)
if not d:
    stats, season_df, player_df = analisis_temporada_equipo_lpf(EQUIPO_DETALLE, SEASON)
else:
    stats, season_df, player_df = d['stats'], d['season_df'], d['player_df']

if stats:
    print(f"=== {nombre} · Temporada {SEASON} ===")
    print(f"{stats['partidos']} PJ | {stats['victorias']}V-{stats['empates']}E-{stats['derrotas']}D "
          f"| {stats['puntos']} pts")
    print(f"Goles: {stats['goles_favor']} a favor / {stats['goles_contra']} en contra")
    print(f"xG: {stats['xg_favor']} favor / {stats['xg_contra']} contra")
    print(f"Local: {stats['rendimiento_local']}  |  Visitante: {stats['rendimiento_visitante']}\n")
    print(season_df[['date', 'opponent', 'venue', 'result', 'chaco_goals', 'rival_goals']]
          .rename(columns={'chaco_goals': 'goles', 'rival_goals': 'goles_rival'})
          .to_string(index=False))

    if not player_df.empty:
        print('\n=== Goleadores ===')
        print(top_scorers(player_df).to_string(index=False))
        print('\n=== Mas utilizados ===')
        print(most_used_players(player_df).to_string(index=False))

    plots_dir = config.lpf_team_plots_dir(EQUIPO_DETALLE, SEASON)
    for f in ['puntos_acumulados.png', 'xg_temporada.png', 'forma_reciente.png',
              'local_visitante.png', 'shotmap_temporada.png', 'top_minutos.png',
              'top_rating.png', 'disciplina.png', 'amenaza_por_situacion.png',
              'markov_estado_marcador.png', 'goles_por_ventana.png']:
        show(os.path.join(plots_dir, f))